In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
from copy import deepcopy
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

1. IMPORT FILES & BASIC EXTRACTION

In [65]:
sales = pd.read_csv('sales.csv', parse_dates=['Date'])
sales_test = pd.read_csv('sample_submission.csv', parse_dates=['Date'])
promotions = pd.read_csv('promotions.csv', parse_dates=['start_date', 'end_date'])

sales['is_train'] = 1
sales_test['is_train'] = 0
if 'Revenue' not in sales_test.columns:
    sales_test['Revenue'] = np.nan
    sales_test['COGS'] = np.nan
df = pd.concat([sales, sales_test]).sort_values('Date').reset_index(drop=True)

2. MERGE

In [66]:
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv('order_items.csv')
geo = pd.read_csv('geography.csv')
products = pd.read_csv('products.csv')

order_items['item_revenue'] = order_items['quantity'] * order_items['unit_price']
merged = order_items.merge(orders[['order_id', 'order_date', 'zip']], on='order_id', how='left')
merged = merged.merge(geo[['zip', 'region']], on='zip', how='left')
merged = merged.merge(products[['product_id', 'category']], on='product_id', how='left')

east_sales = merged[merged['region'] == 'East'].groupby('order_date')['item_revenue'].sum().reset_index(name='East_Revenue')
east_sales.rename(columns={'order_date': 'Date'}, inplace=True)
df = df.merge(east_sales, on='Date', how='left')

sw_sales = merged[merged['category'] == 'Streetwear'].groupby('order_date')['item_revenue'].sum().reset_index(name='Streetwear_Rev')
sw_sales.rename(columns={'order_date': 'Date'}, inplace=True)
df = df.merge(sw_sales, on='Date', how='left')

# WEB TRAFFIC
web_traffic = pd.read_csv('web_traffic.csv', parse_dates=['date'])
web_traffic.rename(columns={'date': 'Date'}, inplace=True)
traffic_pivot = web_traffic.pivot_table(index='Date', columns='traffic_source', values='sessions', aggfunc='sum').fillna(0)
traffic_pivot['Total_Sessions'] = traffic_pivot.sum(axis=1)
df = df.merge(traffic_pivot, on='Date', how='left')

# RETURNS
returns = pd.read_csv('returns.csv', parse_dates=['return_date'])
returns.rename(columns={'return_date': 'Date'}, inplace=True)
daily_returns = returns.groupby('Date')['refund_amount'].sum().reset_index()
df = df.merge(daily_returns, on='Date', how='left')

# INVENTORY
inventory = pd.read_csv('inventory.csv')
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
inv_monthly = inventory.groupby(['year', 'month'])[['stockout_days', 'overstock_flag']].mean().reset_index()
inv_monthly.rename(columns={'stockout_days': 'macro_stockout_days', 'overstock_flag': 'macro_overstock_rate'}, inplace=True)

df = df.merge(inv_monthly, on=['year', 'month'], how='left')
df['macro_stockout_days'] = df['macro_stockout_days'].ffill().fillna(0)
df['macro_overstock_rate'] = df['macro_overstock_rate'].ffill().fillna(0)

print("Data successfully loaded and combined.")

Data successfully loaded and combined.


3. FEATURE ENGINEERING

In [ ]:
df['day'] = df['Date'].dt.day
df['dayofweek'] = df['Date'].dt.dayofweek
df['doy'] = df['Date'].dt.dayofyear
df['week'] = df['Date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['Date'].dt.quarter
df['is_weekend'] = np.where(df['dayofweek'].isin([5, 6]), 1, 0)
df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
df['is_quarter_end'] = df['Date'].dt.is_quarter_end.astype(int)
# df['is_payday'] = df['day'].isin([1, 15, 25, 30, 31]).astype(int)
df['is_double_day'] = (df['day'] == df['month']).astype(int)

# MONTH & QUARTER
df['is_late_month'] = (df['day'] >= 25).astype(int)
df['days_to_month_end'] = df['Date'].dt.daysinmonth - df['day']
df['is_q2'] = (df['quarter'] == 2).astype(int)
df['is_q4'] = (df['quarter'] == 4).astype(int)

# SEASONAL SPIKES
df['is_march_end_to_april_start'] = (((df['month'] == 3) & (df['day'] >= 25)) | ((df['month'] == 4) & (df['day'] <= 5))).astype(int)
df['is_jul_aug_month_end_spike'] = (df['month'].isin([7, 8]) & (df['day'] >= 25)).astype(int)

# WEEKDAY STRENGTH
df['is_monday'] = (df['dayofweek'] == 0).astype(int)
df['is_friday'] = (df['dayofweek'] == 4).astype(int)
df['is_sunday'] = (df['dayofweek'] == 6).astype(int)

# CYCLICAL FEATURES
for k in range(1, 7):
    df[f'year_sin_{k}'] = np.sin(2 * np.pi * k * df['doy'] / 365.25)
    df[f'year_cos_{k}'] = np.cos(2 * np.pi * k * df['doy'] / 365.25)
for k in range(1, 3):
    df[f'week_sin_{k}'] = np.sin(2 * np.pi * k * df['dayofweek'] / 7)
    df[f'week_cos_{k}'] = np.cos(2 * np.pi * k * df['dayofweek'] / 7)
    df[f'month_sin_{k}'] = np.sin(2 * np.pi * k * df['day'] / 31)
    df[f'month_cos_{k}'] = np.cos(2 * np.pi * k * df['day'] / 31)

# ACTUAL PROMOS
df['actual_promos'] = 0
for _, promo in promotions.iterrows():
    mask = (df['Date'] >= promo['start_date']) & (df['Date'] <= promo['end_date'])
    df.loc[mask, 'actual_promos'] += 1

# PROJECTED PROMOS
df['projected_promos'] = df['actual_promos'].copy()
annual_campaigns = [('spring', 3, 18, 4, 17), ('mid_year', 6, 23, 7, 22), ('fall', 8, 30, 10, 1), ('year_end', 11, 18, 1, 2)]
odd_year_campaigns = [('rural_special', 1, 30, 3, 1), ('urban_blowout', 7, 30, 9, 2)]

for year in [2023, 2024]:
    for name, sm, sd, em, ed in annual_campaigns:
        start = pd.Timestamp(year=year, month=sm, day=sd)
        end = pd.Timestamp(year=year if em >= sm else year + 1, month=em, day=ed)
        df.loc[(df['Date'] >= start) & (df['Date'] <= end), 'projected_promos'] += 1
    if year % 2 == 1:
        for name, sm, sd, em, ed in odd_year_campaigns:
            df.loc[(df['Date'] >= pd.Timestamp(year=year, month=sm, day=sd)) & (df['Date'] <= pd.Timestamp(year=year, month=em, day=ed)), 'projected_promos'] += 1

# CUSTOM FEATURES
df['is_fixed_promo_day'] = (df['projected_promos'] > 0).astype(int)
df['death_spiral_index_30d'] = df['is_fixed_promo_day'].rolling(window=30, min_periods=1).sum()

fixed_promo_ends = promotions[promotions['promo_type'].str.lower() == 'fixed']['end_date'].unique()
df['return_wave_echo'] = 0
for end_date in fixed_promo_ends:
    mask = (df['Date'] >= (end_date + pd.Timedelta(days=7))) & (df['Date'] <= (end_date + pd.Timedelta(days=14)))
    df.loc[mask, 'return_wave_echo'] = 1

df['is_liberation_labour'] = (((df['month'] == 4) & (df['day'] >= 28)) | ((df['month'] == 5) & (df['day'] <= 3))).astype(int)
df['is_june_start'] = ((df['month'] == 6) & (df['day'] <= 3)).astype(int)
df['is_sep_2'] = ((df['month'] == 9) & (df['day'] == 2)).astype(int)
df['is_dec_28'] = ((df['month'] == 12) & (df['day'] == 28)).astype(int)

# SPLIT TRAIN/TESTS
train_df = df[df['is_train'] == 1].copy().reset_index(drop=True)
test_df = df[df['is_train'] == 0].copy().reset_index(drop=True)

# Define Feature Sets
base_cols = [
    'year', 'month', 'day', 'dayofweek', 'doy', 'quarter', 'week', 'is_weekend', 
    'is_month_end', 'is_quarter_end', 'is_double_day',
    'year_sin_1', 'year_cos_1', 'year_sin_2', 'year_cos_2', 'year_sin_3', 'year_cos_3',
    'year_sin_4', 'year_cos_4', 'year_sin_5', 'year_cos_5', 'year_sin_6', 'year_cos_6',
    'week_sin_1', 'week_cos_1', 'week_sin_2', 'week_cos_2',
    'month_sin_1', 'month_cos_1', 'month_sin_2', 'month_cos_2'
]   
legacy_features = base_cols + ['actual_promos']
enhanced_features = base_cols + [
    'projected_promos'
    , 'is_fixed_promo_day'
    , 'death_spiral_index_30d'
    , 'return_wave_echo'
    # , 'is_liberation_labour' (3)
    , 'is_june_start'
    # , 'is_sep_2' (1)
    , 'is_dec_28' 
    , 'macro_stockout_days' 
    # , 'macro_overstock_rate' (2)

    # , 'is_late_month' (4)
    , 'days_to_month_end'
    , 'is_q2'
    , 'is_q4'
    , 'is_march_end_to_april_start'
    # , 'is_jul_aug_month_end_spike'
    , 'is_monday'
    , 'is_friday'
    , 'is_sunday'
]

print(f"Base Features prepared. Enhanced features length: {len(enhanced_features)}")

Base Features prepared. Enhanced features length: 45


4. LEAK-FREE PROFILING HELPER

In [183]:
def add_target_profiles(X_train_fold, y_train_fold, X_val_fold, target_name='Revenue'):
    temp = X_train_fold[['doy', 'month', 'dayofweek']].copy()
    temp['target'] = y_train_fold
    
    doy_prof = temp.groupby('doy')['target'].mean().reset_index(name=f'doy_{target_name}_mean')
    mdow_prof = temp.groupby(['month', 'dayofweek'])['target'].mean().reset_index(name=f'mdow_{target_name}_mean')
    global_mean = temp['target'].mean()
    
    def apply_profs(df):
        out = df.copy()
        out = out.merge(doy_prof, on='doy', how='left').merge(mdow_prof, on=['month', 'dayofweek'], how='left')
        out[f'doy_{target_name}_mean'] = out[f'doy_{target_name}_mean'].fillna(global_mean)
        out[f'mdow_{target_name}_mean'] = out[f'mdow_{target_name}_mean'].fillna(global_mean)
        return out
    
    return apply_profs(X_train_fold), apply_profs(X_val_fold)

5. OPTUNA

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
X_train_full = train_df.copy()
y_rev_full = train_df['Revenue']

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42, 'verbose': -1
    }
    scores = []
    for tr_idx, val_idx in tscv.split(X_train_full):
        X_tr, y_tr = X_train_full.iloc[tr_idx], y_rev_full.iloc[tr_idx]
        X_va, y_va = X_train_full.iloc[val_idx], y_rev_full.iloc[val_idx]
        
        X_tr_prof, X_va_prof = add_target_profiles(X_tr, y_tr, X_va)
        train_cols = enhanced_features + ['doy_Revenue_mean', 'mdow_Revenue_mean']
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr_prof[train_cols], np.log1p(y_tr))
        scores.append(np.sqrt(mean_squared_error(y_va, np.expm1(model.predict(X_va_prof[train_cols])))))
    return np.mean(scores)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'random_state': 42, 'verbosity': 0
    }
    scores = []
    for tr_idx, val_idx in tscv.split(X_train_full):
        X_tr, y_tr = X_train_full.iloc[tr_idx], y_rev_full.iloc[tr_idx]
        X_va, y_va = X_train_full.iloc[val_idx], y_rev_full.iloc[val_idx]
        
        X_tr_prof, X_va_prof = add_target_profiles(X_tr, y_tr, X_va)
        train_cols = enhanced_features + ['doy_Revenue_mean', 'mdow_Revenue_mean']
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr_prof[train_cols], np.log1p(y_tr))
        scores.append(np.sqrt(mean_squared_error(y_va, np.expm1(model.predict(X_va_prof[train_cols])))))
    return np.mean(scores)

def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 5000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_state': 42, 'verbose': 0
    }
    scores = []
    for tr_idx, val_idx in tscv.split(X_train_full):
        X_tr, y_tr = X_train_full.iloc[tr_idx], y_rev_full.iloc[tr_idx]
        X_va, y_va = X_train_full.iloc[val_idx], y_rev_full.iloc[val_idx]
        
        X_tr_prof, X_va_prof = add_target_profiles(X_tr, y_tr, X_va)
        train_cols = enhanced_features + ['doy_Revenue_mean', 'mdow_Revenue_mean']
        
        model = CatBoostRegressor(**params)
        model.fit(X_tr_prof[train_cols], np.log1p(y_tr))
        scores.append(np.sqrt(mean_squared_error(y_va, np.expm1(model.predict(X_va_prof[train_cols])))))
    return np.mean(scores)

n_trials = 15

print("--- Tuning LightGBM ---")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=n_trials)

print("\n--- Tuning XGBoost ---")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=n_trials)

print("\n--- Tuning CatBoost ---")
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=n_trials)

[I 2026-05-01 00:28:59,007] A new study created in memory with name: no-name-b096e43d-a64f-4d60-af47-5e4b905561d2


--- Tuning LightGBM ---


[I 2026-05-01 00:29:00,545] Trial 0 finished with value: 1357485.14424977 and parameters: {'n_estimators': 738, 'learning_rate': 0.016159359671071568, 'max_depth': 6, 'num_leaves': 74, 'subsample': 0.8005127489846245, 'colsample_bytree': 0.9406233877040949}. Best is trial 0 with value: 1357485.14424977.
[I 2026-05-01 00:29:03,445] Trial 1 finished with value: 1335172.0196676578 and parameters: {'n_estimators': 3469, 'learning_rate': 0.013548223027908809, 'max_depth': 3, 'num_leaves': 47, 'subsample': 0.9489373348035974, 'colsample_bytree': 0.6532706144052599}. Best is trial 1 with value: 1335172.0196676578.
[I 2026-05-01 00:29:07,323] Trial 2 finished with value: 1370854.421117965 and parameters: {'n_estimators': 1572, 'learning_rate': 0.02365253949202117, 'max_depth': 8, 'num_leaves': 84, 'subsample': 0.9753413394223047, 'colsample_bytree': 0.9336020615005128}. Best is trial 1 with value: 1335172.0196676578.
[I 2026-05-01 00:29:13,991] Trial 3 finished with value: 1365731.9537547373 a


--- Tuning XGBoost ---


[I 2026-05-01 00:31:14,086] Trial 0 finished with value: 1338551.2479680986 and parameters: {'n_estimators': 1874, 'learning_rate': 0.024939730796324975, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.7756677307327325, 'colsample_bytree': 0.9100557476605531, 'gamma': 0.0714444089997424}. Best is trial 0 with value: 1338551.2479680986.
[I 2026-05-01 00:31:24,322] Trial 1 finished with value: 1356535.181622055 and parameters: {'n_estimators': 1610, 'learning_rate': 0.011638140180496751, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.8397815458365706, 'colsample_bytree': 0.9715831751850874, 'gamma': 1.1611242267082188e-06}. Best is trial 0 with value: 1338551.2479680986.
[I 2026-05-01 00:31:33,885] Trial 2 finished with value: 1349118.9948963465 and parameters: {'n_estimators': 3174, 'learning_rate': 0.02309952280618801, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.8355473851573295, 'colsample_bytree': 0.7249236463575603, 'gamma': 1.7735056012631257e-08}. Best is t


--- Tuning CatBoost ---


[I 2026-05-01 00:32:51,256] Trial 0 finished with value: 1347693.983360953 and parameters: {'iterations': 1239, 'learning_rate': 0.02683499654792497, 'depth': 5, 'l2_leaf_reg': 3.0686269212510733, 'random_strength': 2.8900213345040866, 'bagging_temperature': 0.4363141438666759}. Best is trial 0 with value: 1347693.983360953.
[I 2026-05-01 00:33:12,311] Trial 1 finished with value: 1380955.3577714677 and parameters: {'iterations': 871, 'learning_rate': 0.020380889922098085, 'depth': 8, 'l2_leaf_reg': 0.029426801795642037, 'random_strength': 0.051784201157478935, 'bagging_temperature': 0.4165247363280945}. Best is trial 0 with value: 1347693.983360953.
[I 2026-05-01 00:33:38,834] Trial 2 finished with value: 1364947.4132814142 and parameters: {'iterations': 3510, 'learning_rate': 0.016684490153477696, 'depth': 5, 'l2_leaf_reg': 0.007352532012299021, 'random_strength': 2.2326925863139784, 'bagging_temperature': 0.9244867709096962}. Best is trial 0 with value: 1347693.983360953.
[I 2026-05

6. ENSEMBLE

In [180]:
model_factories = {
    'lgb': lambda: lgb.LGBMRegressor(**study_lgb.best_params, random_state=42, verbose=-1),
    'xgb': lambda: xgb.XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0),
    'cat': lambda: CatBoostRegressor(**study_cat.best_params, random_state=42, verbose=0)
}

def train_cv_blended_target(target_col):
    keys = [(f, m) for f in ['legacy', 'enhanced'] for m in ['lgb', 'xgb', 'cat']]
    oof_preds = {k: np.zeros(len(train_df)) for k in keys}
    test_preds = {k: np.zeros(len(test_df)) for k in keys}
    val_indices_list = []

    print(f"\n=== TRAINING 6-MODEL ENSEMBLE ({target_col.upper()}) ===")
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(train_df), 1):
        X_tr_fold = train_df.iloc[tr_idx]
        y_tr_fold = train_df[target_col].iloc[tr_idx]
        X_va_fold = train_df.iloc[val_idx]
        y_va_fold = train_df[target_col].iloc[val_idx]
        val_indices_list.extend(val_idx)
        
        # Dynamic Profiling mapped to the specific target
        X_tr_prof, X_va_prof = add_target_profiles(X_tr_fold, y_tr_fold, X_va_fold, target_name=target_col)    
        _, X_test_prof = add_target_profiles(train_df, train_df[target_col], test_df, target_name=target_col) 
        
        enh_cols = enhanced_features + [f'doy_{target_col}_mean', f'mdow_{target_col}_mean']
        
        for feature_set in ['legacy', 'enhanced']:
            cols = legacy_features if feature_set == 'legacy' else enh_cols
            X_tr = X_tr_prof[cols] if feature_set == 'enhanced' else X_tr_fold[cols]
            X_va = X_va_prof[cols] if feature_set == 'enhanced' else X_va_fold[cols]
            X_te = X_test_prof[cols] if feature_set == 'enhanced' else test_df[cols]
            
            for model_name, factory in model_factories.items():
                key = (feature_set, model_name)
                model = factory()
                
                model.fit(X_tr, np.log1p(y_tr_fold))
                
                oof_preds[key][val_idx] = np.expm1(model.predict(X_va))
                test_preds[key] += np.expm1(model.predict(X_te)) / tscv.get_n_splits()

    # --- SciPy Blending ---
    valid_idx = np.unique(val_indices_list)
    y_valid_true = train_df[target_col].iloc[valid_idx].values
    
    oof_matrix = np.vstack([oof_preds[k][valid_idx] for k in keys])
    def rmse_ensemble(weights):
        blend = weights @ oof_matrix
        return np.sqrt(mean_squared_error(y_valid_true, blend))
        
    constraints = {'type': 'eq', 'fun': lambda w: 1 - sum(w)}
    bounds = [(0, 1)] * len(keys)
    initial_weights = np.ones(len(keys)) / len(keys)
    
    res = minimize(rmse_ensemble, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    
    print(f"\n--- OPTIMAL WEIGHTS ({target_col}) ---")
    for key, weight in zip(keys, res.x):
        if weight > 0.005:
            print(f"  {key[0][:3]}-{key[1]}: {weight:.4f}")
    print(f"FINAL CV RMSE: {res.fun:,.0f}")
    
    final_pred = np.zeros(len(test_df))
    for key, weight in zip(keys, res.x):
        final_pred += weight * test_preds[key]
        
    return final_pred

7. SCIPY BLENDING

In [181]:
# 1. Run the Ensemble for both targets!
final_revenue_pred = train_cv_blended_target('Revenue')
direct_cogs_pred = train_cv_blended_target('COGS')

# 2. Apply the COGS Guardrail
ratio_train = train_df['COGS'] / (train_df['Revenue'] + 1e-5)
ratio_low, ratio_high = ratio_train.quantile(0.005), ratio_train.quantile(0.995)

ratio_pred = direct_cogs_pred / np.maximum(final_revenue_pred, 1.0)
ratio_pred_clipped = np.clip(ratio_pred, ratio_low, ratio_high)
guardrail_cogs_pred = final_revenue_pred * ratio_pred_clipped

# 80/20 Blend protects against edge cases
final_cogs_pred = np.clip((0.80 * direct_cogs_pred) + (0.20 * guardrail_cogs_pred), 0, None)

# 3. Export
submission = pd.DataFrame({
    'Date': test_df['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': np.maximum(0, final_revenue_pred),
    'COGS': np.maximum(0, final_cogs_pred)
})

submission.to_csv('sub59.csv', index=False)
print("\nDone!")
print(f"Final COGS/Revenue ratio: {submission['COGS'].sum() / submission['Revenue'].sum():.6f}")
display(submission.head())


=== TRAINING 6-MODEL ENSEMBLE (REVENUE) ===

--- OPTIMAL WEIGHTS (Revenue) ---
  leg-xgb: 0.6711
  leg-cat: 0.0819
  enh-lgb: 0.0110
  enh-xgb: 0.2358
FINAL CV RMSE: 1,339,049

=== TRAINING 6-MODEL ENSEMBLE (COGS) ===

--- OPTIMAL WEIGHTS (COGS) ---
  leg-xgb: 0.4474
  leg-cat: 0.2926
  enh-xgb: 0.2598
FINAL CV RMSE: 1,136,350

Done!
Final COGS/Revenue ratio: 0.828801


,Date,Revenue,COGS
0,2023-01-01,3.932299e+06,3.373997e+06
1,2023-01-02,1.627003e+06,1.321876e+06
2,2023-01-03,1.592753e+06,1.282712e+06
3,2023-01-04,1.532369e+06,1.222907e+06
4,2023-01-05,1.601827e+06,1.319135e+06
